# Metoda Beam Search pro úlohu O-CVRPTW

Tento notebook implementuje metodu Beam Search (BS) adaptovanou na selektivní kapacitní rozvozní úlohu s časovými okny (O-CVRPTW) dle kapitoly 1.7 diplomové práce. Metoda je deterministická konstruktivní, založená na omezeném prohledávání stavového stromu s šířkou svazku ω.

**Struktura notebooku:**
1. Konfigurace parametrů a načtení datasetu (shodné s MILP notebookem)
2. Evaluátor účelové funkce dle rovnice (8)
3. Parametrizace Beam Search
4. Inkrementální evaluace a seznam kandidátů
5. Konstrukce trasy jednoho vozidla a celého řešení
6. Spuštění experimentu a analýza výsledků

**Požadované knihovny:** pandas, numpy, openpyxl

In [1]:
import math
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any, Set

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass(frozen=True)
class Config:
    
    m: int = 3
    Q: int = 13
    unit_demand: int = 1
    service_time: float = 1.0
    speed: float = 1.0

    PI_LATE: float = 1.5
    P_SKIP: float = 18.0

    # TW generátor
    tw_horizon: float = 120.0
    tw_width: float = 20.0
    tw_a_min: float = 0.0
    tw_a_max: float = 80.0

cfg = Config()
print(cfg)


Config(m=3, Q=13, unit_demand=1, service_time=1.0, speed=1.0, PI_LATE=1.5, P_SKIP=18.0, tw_horizon=120.0, tw_width=20.0, tw_a_min=0.0, tw_a_max=80.0)


## 1. Načtení datasetu a příprava vstupních dat

Načtení datového souboru small_dataset.xlsx (Příloha 1), výpočet euklidovské vzdálenostní matice d_ij, cestovních časů t_ij = d_ij (při speed = 1,0 dle rovnice 9), jednotkové poptávky q_i = 1 a generování časových oken [e_i, l_i] se seedem SEED = 42 dle tabulky 2.2 diplomové práce. Postup je shodný s MILP (Příloha 2), HS (Příloha 3) a ACO (Příloha 4) notebooky.

In [2]:
# Načtení datasetu

# Cesta k datasetu (nutnost upravit podle aktuálního uložení souboru)
DATA_PATH = "small_dataset.xlsx"
df = pd.read_excel(DATA_PATH)

cols_lower = [c.lower() for c in df.columns]
x_col = None
y_col = None
for cand in ["x", "coord_x", "lon", "longitude"]:
    if cand in cols_lower:
        x_col = df.columns[cols_lower.index(cand)]
        break
for cand in ["y", "coord_y", "lat", "latitude"]:
    if cand in cols_lower:
        y_col = df.columns[cols_lower.index(cand)]
        break

if x_col is None or y_col is None:
    raise ValueError(f"Nelze najít sloupce souřadnic. Sloupce v souboru: {list(df.columns)}")

coords = df[[x_col, y_col]].to_numpy(dtype=float)
n = coords.shape[0]
print("Počet uzlů n =", n, "(včetně depa; depo = uzel 0)")

# Matice eukleidovských vzdáleností
d = np.zeros((n, n), dtype=float)
for i in range(n):
    for j in range(n):
        if i != j:
            dx = coords[i, 0] - coords[j, 0]
            dy = coords[i, 1] - coords[j, 1]
            d[i, j] = math.hypot(dx, dy)

# Rychlost = 1 => čas = vzdálenost
t = d / cfg.speed

# Jednotkové poptávky (depot = 0)
q = np.zeros(n, dtype=int)
q[1:] = cfg.unit_demand

# Servisní časy (depot = 0)
service = np.zeros(n, dtype=float)
service[1:] = cfg.service_time

def generate_time_windows(n: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    ready = np.zeros(n, dtype=float)  # a_i (hard)
    due = np.zeros(n, dtype=float)    # b_i (soft)

    ready[0] = 0.0
    due[0] = cfg.tw_horizon

    ready[1:] = rng.uniform(cfg.tw_a_min, cfg.tw_a_max, size=n-1)
    due[1:] = np.minimum(ready[1:] + cfg.tw_width, cfg.tw_horizon)
    return ready, due

ready_time, due_date = generate_time_windows(n, SEED)

print("TW depot:", (ready_time[0], due_date[0]))
print("TW sample customer 1:", (ready_time[1], due_date[1]))


Počet uzlů n = 30 (včetně depa; depo = uzel 0)
TW depot: (0.0, 120.0)
TW sample customer 1: (61.91648388447707, 81.91648388447706)


## 2. Evaluátor účelové funkce

Implementace účelové funkce Z(R) dle rovnice (8) v kapitole 1.3: součet cestovních nákladů, penalizace za zpoždění (γ · T_i^k) a penalizace za neobsloužení (P_skip). Ekvivalence s fitness funkcí f(R) je prokázána v kapitole 1.7 (rovnice 57). Funkce `eval_route` vyhodnocuje jednotlivou trasu, `eval_solution` agreguje přes všechny trasy.

In [3]:
def eval_route(route: List[int]) -> Dict[str, float]:
    """
    Vyhodnocení jedné trasy:
    - travel distance
    - tardiness_sum: součet max(0, start_service - b_i)
    - cap_violation: max(0, load - Q)  (má být 0; hard constraint)
    """
    travel = 0.0
    tardiness_sum = 0.0
    load = sum(q[i] for i in route if i != 0)

    time_now = max(0.0, ready_time[0])
    prev = route[0]

    for node in route[1:]:
        travel += d[prev, node]

        arrival = time_now + service[prev] + t[prev, node]
        start_service = max(arrival, ready_time[node])  # hard lower bound
        tardiness_sum += max(0.0, start_service - due_date[node])  # soft upper

        time_now = start_service
        prev = node

    return {
        "travel": float(travel),
        "tardiness_sum": float(tardiness_sum),
        "cap_violation": float(max(0.0, load - cfg.Q)),
        "load": float(load),
    }


def eval_solution(routes: List[List[int]]) -> Dict[str, Any]:
    served = set()
    for r in routes:
        for node in r:
            if node != 0:
                served.add(node)

    customers = set(range(1, n))
    unserved = customers - served

    travel = 0.0
    tardiness_sum = 0.0
    cap_violation = 0.0
    per_route = []

    for r in routes:
        rr = eval_route(r)
        per_route.append(rr)
        travel += rr["travel"]
        tardiness_sum += rr["tardiness_sum"]
        cap_violation += rr["cap_violation"]

    tardiness_cost = cfg.PI_LATE * tardiness_sum
    skip_cost = cfg.P_SKIP * len(unserved)

    CAP_PENALTY = 1e6  # pojistka, hard constraint
    cap_cost = CAP_PENALTY * cap_violation

    total = travel + tardiness_cost + skip_cost + cap_cost

    return {
        "total": float(total),
        "travel": float(travel),
        "tardiness_sum": float(tardiness_sum),
        "tardiness_cost": float(tardiness_cost),
        "skip_cost": float(skip_cost),
        "cap_cost": float(cap_cost),
        "cap_violation": float(cap_violation),
        "served_cnt": int(len(served)),
        "unserved_cnt": int(len(unserved)),
        "routes": routes,
        "per_route": per_route,
    }


## 3. Parametry Beam Search

Konfigurace parametrů Beam Search dle kapitoly 2.4 diplomové práce (tabulky 2.13 a 2.14):
- **ω (OMEGA):** šířka svazku: maximální počet stavů uchovávaných v každé úrovni stavového stromu
- **MAX_DEPTH:** maximální hloubka prohledávání, odpovídá kapacitě vozidla Q = 13 při jednotkové poptávce
- **CANDIDATE_LIST_SIZE:** počet nejlepších kandidátů zvažovaných pro rozšíření v každém kroku

Konfigurace se mění mezi jednotlivými běhy přístupem ceteris paribus.

In [4]:
from dataclasses import dataclass

@dataclass(frozen=True)
class BeamParams:
    OMEGA: int = 30                # beam width
    MAX_DEPTH: int = 13             # max zákazníků na trase (Q=13, q_i=1)
    CANDIDATE_LIST_SIZE: int = 29   # prakticky bez ořezu pro n=30
    EPS: float = 1e-12

bp = BeamParams()
print(bp)



BeamParams(OMEGA=30, MAX_DEPTH=13, CANDIDATE_LIST_SIZE=29, EPS=1e-12)


## 4. Přípustnost rozšíření a inkrementální náklady

Funkce `try_append` ověřuje přípustnost připojení zákazníka na konec trasy z hlediska kapacitního omezení a tvrdé dolní meze časového okna (podmínka C7). Funkce `incremental_cost` počítá lokální inkrement nákladů Δ = d_ij + γ · ΔL_j pro řazení kandidátů. Funkce `top_candidates` vrací seznam kandidátů seřazený vzestupně dle inkrementálních nákladů.

In [5]:
def try_append(prev: int, cand: int, time_now: float, load_now: int) -> Tuple[bool, float, float]:
    """
    Hard: kapacita + hard lower bound (start >= a_i)
    Soft: horní mez přes tardiness increment
    """
    if cand == 0 or cand == prev:
        return False, time_now, 0.0
    if load_now + q[cand] > cfg.Q:
        return False, time_now, 0.0

    arrival = time_now + service[prev] + t[prev, cand]
    start_service = max(arrival, ready_time[cand])  # hard lower
    tard_inc = max(0.0, start_service - due_date[cand])  # soft upper
    return True, start_service, tard_inc


def incremental_cost(prev: int, cand: int, time_now: float, load_now: int) -> float:
    """
    Lokální inkrement pro candidate pruning:
    Δ = distance + PI_LATE * tardiness_increment
    """
    feasible, _, tard_inc = try_append(prev, cand, time_now, load_now)
    if not feasible:
        return float("inf")
    return d[prev, cand] + cfg.PI_LATE * tard_inc


def top_candidates(prev: int, remaining: List[int], time_now: float, load_now: int) -> List[int]:
    scored = []
    for cand in remaining:
        inc = incremental_cost(prev, cand, time_now, load_now)
        if math.isfinite(inc):
            scored.append((inc, cand))
    scored.sort()  # min inc first
    return [c for _, c in scored[:bp.CANDIDATE_LIST_SIZE]]


## 5. Beam Search pro konstrukci jedné trasy

Implementace prohledávání stavového stromu pro jedno vozidlo dle pseudokódu v kapitole 1.7. V každé úrovni je z každého stavu ve svazku vygenerována množina potomků rozšířením o přípustné kandidáty. Potomci jsou ohodnoceni hodnoticí funkcí g(η) dle rovnice (55), která zahrnuje akumulované cestovní náklady, tardiness penalizaci a pesimistický odhad penalizace za neobsloužené zákazníky. Svazek je ořezán na ω nejlepších stavů. Průběžně je sledována nejlepší uzavřená trasa (s návratem do depota) hodnocená konzistentně s g(η).

In [6]:
from dataclasses import dataclass
from typing import List, Set, Tuple

@dataclass
class State:
    route: List[int]
    visited: Set[int]
    prev: int
    time_now: float
    load_now: int
    dist: float
    tard: float
    score: float  # ranking (nižší = lepší)


def beam_search_one_vehicle(remaining_set: Set[int], vehicle_index: int) -> Tuple[List[int], int]:
    """
    Beam Search konstruuje jednu trasu pro vozidlo `vehicle_index` (0-based).
    Vrací (trasa, hloubka_nalezení) – hloubku, ve které bylo naposledy zlepšeno nejlepší uzavřené řešení.
    Ranking i uzavírání trasy zahrnuje penalizaci za (odhad) neobsloužených zákazníků, aby algoritmus nebyl motivován konstruovat pouze [0, i, 0].
    """
    vehicles_left_after = cfg.m - (vehicle_index + 1)

    init = State(
        route=[0],
        visited=set(),
        prev=0,
        time_now=max(0.0, ready_time[0]),
        load_now=0,
        dist=0.0,
        tard=0.0,
        score=0.0,
    )

    beam: List[State] = [init]

    # nejlepší uzavřená trasa pro toto vozidlo (hodnocena konzistentně se score)
    best_closed_route = [0, 0]
    best_closed_score = float("inf")
    best_closed_depth = 0  # hloubka, ve které bylo nalezeno nejlepší uzavřené řešení

    max_depth = min(bp.MAX_DEPTH, len(remaining_set))

    for depth in range(max_depth):
        offspring: List[State] = []

        for st in beam:
            rem_list = list(remaining_set - st.visited)
            if not rem_list:
                continue

            # kandidáti: (prakticky bez ořezu; řazeno dle inkrementu distance + PI*tard_inc)
            cand_list = top_candidates(st.prev, rem_list, st.time_now, st.load_now)

            for cand in cand_list:
                feasible, new_time, tard_inc = try_append(st.prev, cand, st.time_now, st.load_now)
                if not feasible:
                    continue

                new_route = st.route + [cand]
                new_visited = set(st.visited)
                new_visited.add(cand)

                new_dist = st.dist + d[st.prev, cand]
                new_tard = st.tard + tard_inc
                new_load = st.load_now + int(q[cand])

                # ODHAD UNOBSLOUŽENÝCH = pesimistický tlak na obsluhu
                # pokud bychom řešení uzavřeli "teď", pak zbytek zákazníků zůstane unserved
                remaining_after = len(remaining_set) - len(new_visited)
                unserved_hat = remaining_after

                # ranking criterion konzistentní s cílovou funkcí
                new_score = new_dist + cfg.PI_LATE * new_tard + cfg.P_SKIP * unserved_hat

                offspring.append(State(
                    route=new_route,
                    visited=new_visited,
                    prev=cand,
                    time_now=new_time,
                    load_now=new_load,
                    dist=new_dist,
                    tard=new_tard,
                    score=new_score
                ))

        if not offspring:
            break

        # beam pruning
        offspring.sort(key=lambda s: s.score)
        beam = offspring[:bp.OMEGA]

        # průběžně zvažuje uzavření trasy návratem do depa, ale hodnotí konzistentně se score
        for st in beam:
            closed_route = st.route + [0]
            rr = eval_route(closed_route)

            if rr["cap_violation"] > 0.0:
                continue

            # z pohledu uzavření: stále platí pesimistický odhad unserved_hat = zbývající zákazníci
            remaining_after_closed = len(remaining_set) - len(st.visited)
            unserved_hat_closed = remaining_after_closed

            closed_score = rr["travel"] + cfg.PI_LATE * rr["tardiness_sum"] + cfg.P_SKIP * unserved_hat_closed

            if closed_score < best_closed_score:
                best_closed_score = closed_score
                best_closed_route = closed_route
                best_closed_depth = depth + 1  # zaznamenání hloubky zlepšení (1-based)

    return best_closed_route, best_closed_depth

## 6. Konstrukce celého řešení a validace

Sekvenční konstrukce řešení pro m = 3 vozidla: každé vozidlo konstruuje trasu metodou Beam Search (sekce 5) z množiny dosud neobsloužených zákazníků. Zákazníci obsloužení předchozími vozidly jsou z množiny odebráni. Funkce `is_solution_valid` ověřuje konzistenci výsledného řešení (depoty, unikátnost obsluhy, kapacitní omezení, tvrdá dolní mez časových oken).

In [7]:
def construct_solution_beam() -> Tuple[List[List[int]], List[int]]:
    remaining_set: Set[int] = set(range(1, n))
    routes: List[List[int]] = []
    depths: List[int] = []  # hloubka nalezení nejlepší trasy pro každé vozidlo

    for k in range(cfg.m):
        route_k, depth_k = beam_search_one_vehicle(remaining_set, vehicle_index=k)

        # odebere obsloužené zákazníky z remaining
        for node in route_k:
            if node != 0:
                remaining_set.discard(node)

        routes.append(route_k)
        depths.append(depth_k)

    return routes, depths


def is_solution_valid(routes: List[List[int]]) -> bool:
    # depot start/end
    for r in routes:
        if len(r) < 2 or r[0] != 0 or r[-1] != 0:
            return False

    # uniqueness
    seen = set()
    for r in routes:
        for node in r:
            if node == 0:
                continue
            if node in seen:
                return False
            seen.add(node)

    # hard capacity
    for r in routes:
        if sum(q[i] for i in r if i != 0) > cfg.Q:
            return False

    # hard lower TW vynuceno max() v simulaci, validuje skrz simulaci
    for r in routes:
        time_now = max(0.0, ready_time[0])
        prev = r[0]
        for node in r[1:]:
            arrival = time_now + service[prev] + t[prev, node]
            start_service = max(arrival, ready_time[node])
            if start_service + 1e-9 < ready_time[node]:
                return False
            time_now = start_service
            prev = node

    return True

## 7. Spuštění experimentu

Spuštění Beam Search s konfigurací parametrů dle kapitoly 2.4 diplomové práce. Metoda je deterministická a každá konfigurace je vyhodnocena jediným během (tabulka 2.6).

In [8]:
start_time = time.perf_counter()

routes, depths = construct_solution_beam()

runtime = time.perf_counter() - start_time

if not is_solution_valid(routes):
    # bezpečnostní fallback (nemělo by nastat)
    routes = [[0, 0] for _ in range(cfg.m)]
    depths = [0] * cfg.m

## 8. Souhrnné výsledky

Rozklad účelové funkce Z(R) na jednotlivé složky dle rovnice (8): cestovní náklady dist(R), penalizace za zpoždění γ · tard(R) a penalizace za neobsloužení P_skip · |C \ served(R)|. Výpis tras jednotlivých vozidel a počtu obsloužených zákazníků pro srovnání s ostatními metodami (kapitola 2.5).

In [9]:
rec = eval_solution(routes)

print("ŘEŠENÍ DOKONČENO (BEAM SEARCH)")
print(f"Runtime: {runtime:.15f} s")
print(f"Objective: {rec['total']:.6f}")
print()

print("Objective breakdown (EXACT):")
print("distance     =", rec["travel"])
print(f"{cfg.PI_LATE}*tardiness=", cfg.PI_LATE * rec["tardiness_sum"])
print(f"{cfg.P_SKIP}*unserved  =", rec["skip_cost"])
print()

print("Objective breakdown (EXACT):")
print("total         =", rec["total"])
print("distance      =", rec["travel"])
print(
    f"{cfg.PI_LATE}*tardiness = {rec['tardiness_cost']:.6f}  "
    f"(tardiness_sum = {rec['tardiness_sum']:.6f})"
)
print(
    f"{cfg.P_SKIP}*unserved   = {rec['skip_cost']:.6f}  "
    f"(unserved = {rec['unserved_cnt']})"
)
print("cap_violation =", rec["cap_violation"])
print()

print("Served customers:", rec["served_cnt"], "/", (n - 1))
print("Routes:")
for k, r in enumerate(rec["routes"], start=1):
    customers_on_route = len(r) - 2  # bez obou depotů
    print(f"Vehicle {k}: {r}  (best closed at depth {depths[k-1]} / {customers_on_route})")

ŘEŠENÍ DOKONČENO (BEAM SEARCH)
Runtime: 0.187801200023387 s
Objective: 135.201564

Objective breakdown (EXACT):
distance     = 125.68376738827737
1.5*tardiness= 9.517796984880334
18.0*unserved  = 0.0

Objective breakdown (EXACT):
total         = 135.2015643731577
distance      = 125.68376738827737
1.5*tardiness = 9.517797  (tardiness_sum = 6.345198)
18.0*unserved   = 0.000000  (unserved = 0)
cap_violation = 0.0

Served customers: 29 / 29
Routes:
Vehicle 1: [0, 29, 22, 9, 19, 20, 4, 7, 25, 8, 24, 12, 3, 6, 0]  (best closed at depth 13 / 13)
Vehicle 2: [0, 28, 15, 16, 21, 17, 1, 23, 14, 0]  (best closed at depth 8 / 8)
Vehicle 3: [0, 18, 5, 2, 11, 26, 10, 27, 13, 0]  (best closed at depth 8 / 8)
